# Causal Inference and AI Final Project

## Historical Newspaper Explanations of Juvenile Delinquency

**Research Question**  
How did historical American newspapers explain the causes of juvenile delinquency, and what causal role was attributed to parental neglect?

This notebook implements the six-step causal inference pipeline used in the final report:

1. Source Text Selection  
2. Cleaning and Normalization  
3. Extraction of Causal Assertions  
4. Knowledge Graph Construction  
5. DAG Construction  
6. Causal Identification and Discursive Effect Estimation


## 0. Setup

Upload `Final_Project.csv` to the same directory as this notebook before running the cells.

The CSV is expected to contain the following columns:

- `ID`
- `Date`
- `Newspaper`
- `Quote`
- `Cause`
- `Cause_Normalized`
- `Effect`
- `Effect_Normalized`
- `Relation`
- `Stance`
- `URL`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)


## Step 1 — Source Text Selection

The corpus consists of 40 newspaper article sentences collected mainly from the Library of Congress Chronicling America archive.  
The time period is 1905–1959. This wide range was selected to avoid overfitting the analysis to a single event or year and to examine repeated patterns in historical causal explanations.

This project follows **Track B**, because it uses a separately selected research question and corpus rather than reproducing the provided example corpus.


In [ ]:
# Load the manually curated dataset
df = pd.read_csv("newspaper_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# Basic column check
expected_columns = [
    "ID", "Date", "Newspaper", "Quote", "Cause", "Cause_Normalized",
    "Effect", "Effect_Normalized", "Relation", "Stance", "URL"
]

missing = [c for c in expected_columns if c not in df.columns]
if missing:
    print("Missing columns:", missing)
else:
    print("All expected columns are present.")

df.columns.tolist()


## Step 2 — Cleaning and Normalization

Historical newspaper texts often describe the same concept using different expressions.  
For example, `disregard of parental duty`, `lack of parental supervision`, and `parental laxity` were normalized as **Parental Neglect**.

The goal of normalization is not to erase the original wording, but to preserve the original quote while making concept-level aggregation possible.


In [ ]:
# Show examples of original causes and normalized causes
normalization_examples = (
    df[["Cause", "Cause_Normalized", "Effect", "Effect_Normalized"]]
    .drop_duplicates()
    .sort_values(["Cause_Normalized", "Cause"])
)

display(normalization_examples.head(20))


In [ ]:
# Count normalized causes
cause_counts = (
    df["Cause_Normalized"]
    .value_counts()
    .reset_index()
)
cause_counts.columns = ["Cause_Normalized", "count"]
display(cause_counts)


## Step 3 — Extraction of Causal Assertions

Each row represents one manually identified causal assertion.  
The relation field distinguishes different types of causal language:

- `CAUSES`: direct causal claim
- `CONTRIBUTES_TO`: partial contribution
- `ATTRIBUTED_TO`: attribution or blame
- `MIXED_CAUSATION`: multiple or ambiguous causes

Example:  
“Nearly every case of juvenile delinquency is caused by lack of proper food”  
was extracted as:

- Cause: Poverty
- Effect: Juvenile Delinquency
- Relation: CAUSES


In [ ]:
# Relation type distribution
relation_counts = (
    df["Relation"]
    .value_counts()
    .reset_index()
)
relation_counts.columns = ["Relation", "count"]
display(relation_counts)


In [ ]:
# Causal assertion table from the text-supported dataset
causal_table = (
    df.groupby(["Cause_Normalized", "Effect_Normalized", "Relation"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(causal_table)


## Step 4 — Knowledge Graph Construction

The knowledge graph is exported as two CSV files:

- `nodes.csv`: concepts and their roles in the causal model
- `edges.csv`: directed relationships between concepts

The graph uses a plain tabular edge-list format, which is sufficient for a small humanities-based causal graph.


In [ ]:
# Build nodes.csv
nodes = pd.DataFrame([
    ["Parental Neglect", "Treatment"],
    ["Juvenile Delinquency", "Outcome"],
    ["Broken Homes", "Confounder Candidate"],
    ["Poverty", "Confounder Candidate"],
    ["Working Mothers", "Upstream Cause"],
    ["Shiftlessness", "Upstream Cause"],
    ["Wartime Conditions", "Upstream Cause"],
    ["Comic Books", "Alternative Explanation"],
    ["Moving Pictures", "Alternative Explanation"],
], columns=["id", "type"])

display(nodes)

nodes.to_csv("nodes.csv", index=False)
print("Saved nodes.csv")


In [ ]:
# Build edges.csv for the final DAG / knowledge graph export

edges = pd.DataFrame([
    ["Shiftlessness", "Parental Neglect", "Text-supported", 1],
    ["Wartime Conditions", "Parental Neglect", "Text-supported", 1],
    ["Working Mothers", "Parental Neglect", "Analyst-added", ""],
    ["Poverty", "Parental Neglect", "Analyst-added", ""],
    ["Broken Homes", "Parental Neglect", "Analyst-added", ""],
    ["Parental Neglect", "Juvenile Delinquency", "Text-supported", 22],
    ["Broken Homes", "Juvenile Delinquency", "Text-supported", 7],
    ["Poverty", "Juvenile Delinquency", "Text-supported", 2],
    ["Comic Books", "Juvenile Delinquency", "Text-supported", 2],
    ["Moving Pictures", "Juvenile Delinquency", "Text-supported", 1],
], columns=["source", "target", "edge_type", "support"])

display(edges)

edges.to_csv("edges.csv", index=False)
print("Saved edges.csv")


## Step 5 — DAG Construction

The main causal question is:

**What is the causal effect of Parental Neglect on Juvenile Delinquency?**

- Treatment: `Parental Neglect`
- Outcome: `Juvenile Delinquency`
- Confounder candidates: `Broken Homes`, `Poverty`

Solid arrows represent text-supported causal assertions.  
Dashed arrows represent analyst-added relationships used to construct a coherent DAG for causal identification.


In [ ]:
# Create final DAG figure

G = nx.DiGraph()

for _, row in edges.iterrows():
    G.add_edge(row["source"], row["target"], edge_type=row["edge_type"], support=row["support"])

pos = {
    "Poverty": (-2.2, 2.0),
    "Working Mothers": (0.0, 2.0),
    "Broken Homes": (2.2, 2.0),
    "Parental Neglect": (0.0, 0.7),
    "Juvenile Delinquency": (0.0, -0.8),
    "Comic Books": (2.6, -0.8),
    "Moving Pictures": (4.0, -0.8),
    "Shiftlessness": (-4.0, 2.0),
    "Wartime Conditions": (-4.0, 0.7),
}

main_nodes = [
    "Poverty", "Working Mothers", "Broken Homes",
    "Parental Neglect", "Juvenile Delinquency"
]

plt.figure(figsize=(12, 7))

# Draw main nodes
nx.draw_networkx_nodes(G, pos, nodelist=main_nodes, node_size=3600, node_color="white", edgecolors="black", linewidths=1.5)
nx.draw_networkx_labels(G, pos, labels={n: n for n in main_nodes}, font_size=11, font_weight="bold")

# Draw text-supported edges in the main DAG
text_edges = [
    ("Parental Neglect", "Juvenile Delinquency"),
    ("Broken Homes", "Juvenile Delinquency"),
]
nx.draw_networkx_edges(
    G, pos, edgelist=text_edges, arrows=True, arrowstyle="-|>", arrowsize=20,
    width=1.8, edge_color="black"
)

# Draw analyst-added edges
analyst_edges = [
    ("Poverty", "Parental Neglect"),
    ("Working Mothers", "Parental Neglect"),
    ("Broken Homes", "Parental Neglect"),
]
nx.draw_networkx_edges(
    G, pos, edgelist=analyst_edges, arrows=True, arrowstyle="-|>", arrowsize=18,
    width=1.5, edge_color="black", style="dashed"
)

# Edge labels
edge_labels = {
    ("Parental Neglect", "Juvenile Delinquency"): "Text-supported\\n(n=22)",
    ("Broken Homes", "Juvenile Delinquency"): "Text-supported\\n(n=7)",
    ("Poverty", "Parental Neglect"): "analyst-added",
    ("Working Mothers", "Parental Neglect"): "analyst-added",
    ("Broken Homes", "Parental Neglect"): "analyst-added",
}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=9, label_pos=0.55)

plt.title(
    "Causal Model of Explanations for Juvenile Delinquency\\nin Historical Newspapers (1905–1959)",
    fontsize=16,
    fontweight="bold"
)

plt.text(
    0, 2.65,
    "Edges are labeled by source type. Numbers indicate frequency of assertions in 40 newspaper articles.",
    ha="center",
    fontsize=11,
    style="italic"
)

# Legend and notes
plt.text(
    3.3, 1.3,
    "Legend\\n\\n"
    "→  Text-supported\\n"
    "     (asserted in newspapers)\\n\\n"
    "⇢  Analyst-added\\n"
    "     (added by the researcher)\\n\\n"
    "Alternative explanations not included\\n"
    "in the main identification model:\\n"
    "Comic Books (n=2), Moving Pictures (n=1)",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor="black")
)

plt.text(
    -3.6, -1.7,
    "Model Summary: Treatment = Parental Neglect | Outcome = Juvenile Delinquency | "
    "Confounder candidates = Broken Homes, Poverty\\n"
    "Note: Shiftlessness and Wartime Conditions were mentioned only once each and were omitted from "
    "the final DAG because they did not materially affect the identification strategy.",
    fontsize=9,
    style="italic"
)

plt.axis("off")
plt.tight_layout()
plt.savefig("causal_dag.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved causal_dag.png")


## Step 6 — Causal Identification

The DAG contains two important backdoor paths:

1. `Parental Neglect ← Broken Homes → Juvenile Delinquency`
2. `Parental Neglect ← Poverty → Juvenile Delinquency`

Therefore, the adjustment set is:

**Adjustment Set = {Broken Homes, Poverty}**

Under the assumptions encoded in the DAG, the total causal effect of parental neglect on juvenile delinquency is identifiable by adjusting for these two variables.


In [ ]:
# Causal identification result

treatment = "Parental Neglect"
outcome = "Juvenile Delinquency"
adjustment_set = ["Broken Homes", "Poverty"]

print("Treatment:", treatment)
print("Outcome:", outcome)
print("Adjustment Set:", adjustment_set)
print()
print("Identification statement:")
print("P(Juvenile Delinquency | do(Parental Neglect)) is identifiable by adjusting for Broken Homes and Poverty.")


## Estimated Discursive Effect

This project does not estimate a real-world causal effect size because the data consist of newspaper causal assertions rather than individual-level observations.

However, the relative prominence of each causal explanation in historical newspaper discourse can be estimated from assertion frequencies.


In [ ]:
# Estimate discursive frequency of Parental Neglect as a causal explanation

discursive_counts = pd.DataFrame([
    ["Parental Neglect", 22],
    ["Broken Homes", 7],
    ["Poverty", 2],
    ["Working Mothers", 2],
    ["Comic Books", 2],
    ["Moving Pictures", 1],
], columns=["Cause", "Frequency"])

total_assertions = discursive_counts["Frequency"].sum()
parental_neglect_count = discursive_counts.loc[
    discursive_counts["Cause"] == "Parental Neglect", "Frequency"
].iloc[0]

discursive_probability = parental_neglect_count / total_assertions

display(discursive_counts)

print("Total causal assertions:", total_assertions)
print("Parental Neglect assertions:", parental_neglect_count)
print(f"P(Parental Neglect) = {parental_neglect_count}/{total_assertions} = {discursive_probability:.3f}")
print(f"Discursive percentage = {discursive_probability*100:.1f}%")


## Final Interpretation

The analysis leads to two conclusions:

1. At the discourse level, parental neglect was the dominant explanation for juvenile delinquency in the collected historical newspaper articles.
2. At the causal-model level, the effect of parental neglect on juvenile delinquency is identifiable under the constructed DAG by adjusting for Broken Homes and Poverty.

The value 61.1% is **not** a real-world causal effect size.  
It represents the proportion of causal assertions in the newspaper discourse that identified parental neglect as a cause of juvenile delinquency.


In [ ]:
# Confirm exported files
from pathlib import Path

for file_name in ["nodes.csv", "edges.csv", "causal_dag.png"]:
    path = Path(file_name)
    print(file_name, "exists:" if path.exists() else "missing")
